# GPU Evaluation Suite: Contrastive Reflexive Memory (CRM)
### Paper: *Reflexive Inhibition in LLM Agents: Overcoming Negative-Association Blindness via Contrastive Failure Masking*

This notebook runs the GPU-accelerated experimental suite on Google Colab (T4 / A100) or Kaggle:
1. **Matched 5-Method Comparison**: `vanilla`, `mem0_rag`, `crm_prompt`, `crm_mask` (real LogitsProcessor), and `crm_mask_redirect` on the exact same Hugging Face Transformers backend.
2. **Full Interference Horizon Ablation**: 160 TEST tasks $\times$ $M \in \{4, 12, 30, 120\}$ across 3 seeds (`[42, 123, 999]`) with real distractor turns inserted and true `prompt_tokens` logged.
3. **Full Paraphrase Stress Test**: 80 classical TEST tasks $\times$ $\{\text{exact}, \text{lexical}, \text{structural}\}$ across all 5 methods and 3 seeds.
4. **Model Scale Generalization**: Qwen2.5-7B-Instruct in 4-bit (`bitsandbytes`).

**Resumability**: Every trial is immediately appended to `gpu_experiments.jsonl` on Google Drive. If disconnected, simply re-run cells to resume from the last trial.

## 1. Environment Setup & Dependency Installation

In [ ]:
# Install dependencies
!pip install -q transformers accelerate bitsandbytes sentence-transformers torch pandas

## 2. Google Drive Mount / Storage Configuration

In [ ]:
import os
import sys

# Check environment (Colab vs Kaggle vs Local)
if 'google.colab' in str(get_ipython()):
    from google.colab import drive
    drive.mount('/content/drive')
    WORK_DIR = '/content/drive/MyDrive/crm-reflexive-inhibition'
    os.makedirs(WORK_DIR, exist_ok=True)
    %cd $WORK_DIR
elif os.path.exists('/kaggle/working'):
    WORK_DIR = '/kaggle/working/crm-reflexive-inhibition'
    os.makedirs(WORK_DIR, exist_ok=True)
    %cd $WORK_DIR
else:
    WORK_DIR = os.path.abspath('.')

print(f"Active working directory: {WORK_DIR}")
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

## 3. Verify CUDA GPU Device

In [ ]:
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 4. Run Matched 5-Method Horizon Suite (Qwen2.5-3B-Instruct)
Evaluates $M \in \{4, 12, 30, 120\}$ across 3 seeds (`[42, 123, 999]`).
Includes real LogitsProcessor token mask (`crm_mask`) and prefix redirection (`crm_mask_redirect`).

In [ ]:
from scripts.run_gpu_experiments import run_gpu_suite

# Execute Qwen2.5-3B-Instruct experiments
run_gpu_suite(
    model_name="Qwen/Qwen2.5-3B-Instruct",
    load_in_4bit=False,
    horizons=[4, 12, 30, 120],
    seeds=[42, 123, 999],
    output_dir="results/raw_logs",
)

## 5. Run Scale Comparison on Qwen2.5-7B-Instruct (4-Bit bitsandbytes)
Evaluates 5 methods on Qwen2.5-7B-Instruct under 4-bit NF4 quantization to assess scaling behavior.

In [ ]:
# Execute Qwen2.5-7B-Instruct (4-bit)
run_gpu_suite(
    model_name="Qwen/Qwen2.5-7B-Instruct",
    load_in_4bit=True,
    horizons=[4, 8, 30],
    seeds=[42, 123, 999],
    output_dir="results/raw_logs",
)

## 6. Run Strict Integrity Gate Checks
Verifies distinct responses > 20%, gate equality, token scaling, zero error rows, and zero hash collisions.

In [ ]:
from checks import run_full_anti_stub_audit

passed = run_full_anti_stub_audit(check_horizon=True)
print(f"Checks Status: {'ALL PASSED' if passed else 'FAILED'}")